> **Disclaimer:** This workshop is provided for educational and informational purposes only. The sample code and configurations are not intended for production use. Please review and adapt them according to your organization's security and compliance requirements. AWS service charges may apply for resources created during this workshop. Video content used in this workshop is sourced from publicly available materials and is attributed where applicable.

# TwelveLabs Marengo on Amazon Bedrock Workshop

TwelveLabs is a leading provider of multimodal AI models specializing in video understanding and analysis. Amazon Bedrock now offers TwelveLabs Marengo Embed, which generates high-quality embeddings for video, text, audio, and image content. This model empowers developers to build applications that can intelligently process, analyze, and derive insights from video data at scale.

TwelveLabs models analyze and combine information from all modalities (visual, audio, text) to accurately interpret videos holistically — similar to how humans watch, listen, and read simultaneously.

Supported modalities:
- **Visual**: Actions, objects, events, OCR, brand logos
- **Audio**: Ambient sounds, music, human speech

## Part 0: Setup

### Dependencies

In [13]:
%pip install -r requirements.txt -Uq


[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import boto3, botocore
import json
import re
import pandas as pd
import numpy as np
import uuid
import time
import base64
from IPython.display import clear_output, HTML, display, Image
from sklearn.metrics.pairwise import cosine_similarity
from opensearchpy import AWSV4SignerAuth, OpenSearch, RequestsHttpConnection

### Configure boto3

In [15]:
# Initialize AWS session
session = boto3.Session() # TODO: (Optional) Replace with your AWS profile, keep as is to use the default profile

In [16]:
# Get AWS Region from session
AWS_REGION = session.region_name
# AWS_REGION = "us-east-1" # OPTIONAL: Manual override for workshop region

print(f"AWS Region: {AWS_REGION}")

workshop_supported_regions = [
    "us-east-1", # N. Virginia
    "eu-west-1", # Ireland
    "ap-northeast-2" # Seoul
]

if AWS_REGION not in workshop_supported_regions:
    raise ValueError(f"AWS Region {AWS_REGION} is not supported for this workshop. Please use one of the following regions: {workshop_supported_regions}")

AWS Region: us-east-1


In [17]:
# Initialize required AWS clients
bedrock_client = session.client('bedrock-runtime')
s3_client = session.client('s3')

### Configure S3 bucket

In [18]:
# S3 Configuration
S3_BUCKET_NAME = "twelvelabs-video-demo-bucket" # TODO: Replace with your S3 bucket name
S3_VIDEOS_PATH = "videos"
S3_IMAGES_PATH = "images"
S3_EMBEDDINGS_PATH = "embeddings"

# Validate S3 bucket name
if S3_BUCKET_NAME == "<YOUR_S3_BUCKET_NAME>" or S3_BUCKET_NAME == "":
    raise ValueError("Please replace <YOUR_S3_BUCKET_NAME> with your S3 bucket name")

## Part 1: Multimodal Embeddings with Marengo

### Part 1a: What is an embedding?

Marengo generates multimodal embeddings — contextual vector representations (512-dim) that capture interactions between visual, audio, and text modalities. These embeddings can be used for similarity search, anomaly detection, recommendations, and RAG systems.

Learn more: [The Multimodal Evolution of Vector Embeddings](https://www.twelvelabs.io/blog/multimodal-embeddings)

In [19]:
# Sample embeddings
sample_embedding_1 = np.random.rand(1, 512)
sample_embedding_2 = np.random.rand(1, 512)

df_embedding_1 = pd.DataFrame(sample_embedding_1)
df_embedding_2 = pd.DataFrame(sample_embedding_2)

df_embedding_1

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,0.750949,0.141556,0.058224,0.728821,0.119008,0.899222,0.749557,0.497119,0.963072,0.115224,...,0.238367,0.598211,0.482758,0.644851,0.968839,0.237959,0.988311,0.111476,0.342857,0.223928


In [20]:
# Sample video embedding
sample_video_embedding = np.random.rand(5, 512)
df_video_embedding = pd.DataFrame(sample_video_embedding)
df_video_embedding

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,0.990833,0.206266,0.953138,0.252980,0.759763,0.047366,0.007991,0.550847,0.148845,0.247597,...,0.579804,0.545977,0.102838,0.575411,0.252549,0.675189,0.806115,0.315842,0.951075,0.989381
1,0.636744,0.535326,0.468921,0.842347,0.226515,0.771743,0.435561,0.966097,0.995474,0.191852,...,0.109812,0.602775,0.806344,0.703627,0.168456,0.678237,0.821732,0.517962,0.077376,0.028043
2,0.174199,0.659740,0.392573,0.071532,0.059165,0.787242,0.405761,0.349405,0.408144,0.611180,...,0.478502,0.035719,0.506729,0.928464,0.993100,0.515964,0.792195,0.181425,0.306013,0.228323
3,0.135314,0.277935,0.278199,0.035185,0.639845,0.994629,0.180137,0.332448,0.538899,0.311227,...,0.614297,0.197943,0.543468,0.059045,0.990019,0.899341,0.587764,0.615750,0.552329,0.245596
4,0.363752,0.690443,0.253896,0.781795,0.788025,0.223280,0.132207,0.382328,0.668328,0.685871,...,0.702788,0.163220,0.073033,0.212533,0.852234,0.934965,0.950653,0.977161,0.239624,0.187500


### Part 1b: Calculating cosine similarity

Cosine similarity measures the similarity between two vectors by calculating the cosine of the angle between them. Values range from -1 (opposite) to 1 (identical). It is scale-invariant and widely used for comparing embeddings.

In [21]:
# Cosine similarity between two single segment embeddings
similarity = cosine_similarity(df_embedding_1, df_embedding_2)
pd.DataFrame(similarity)

,0
0,0.737234


In [22]:
# Cosine similarity with a multi-segment embedding
similarities = cosine_similarity(df_video_embedding, df_embedding_1)
pd.DataFrame(similarities)

,0
0,0.738915
1,0.769812
2,0.741724
3,0.735428
4,0.755221


In [23]:
# Getting the max similarity and the index of the max similarity
max_similarity = np.max(similarities)
max_similarity_index = np.argmax(similarities)

print(f"Max similarity: {max_similarity}")
print(f"Index of max similarity: {max_similarity_index}")

Max similarity: 0.7698124376404144
Index of max similarity: 1


---
## Part 2: Building Multimodal Video Search


### Part 2a: Storing videos in S3

#### Set up sample dataset to S3 bucket

In [24]:
# AWS Account ID for S3 bucket ownership
aws_account_id = session.client('sts').get_caller_identity()["Account"]

print(f"AWS Account ID: {aws_account_id}")
print(f"S3 Bucket: {S3_BUCKET_NAME}")
print(f"S3 Videos Path: {S3_VIDEOS_PATH}")
print(f"S3 Images Path: {S3_IMAGES_PATH}")
print(f"S3 Embeddings Path: {S3_EMBEDDINGS_PATH}")

# Verify bucket access
try:
    s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
    print(f"✅ Successfully connected to S3 bucket: {S3_BUCKET_NAME}")
except Exception as e:
    print(f"❌ Error accessing S3 bucket: {e}")
    print("Please ensure the bucket exists and you have proper permissions.")

AWS Account ID: 087967639890
S3 Bucket: twelvelabs-video-demo-bucket
S3 Videos Path: videos
S3 Images Path: images
S3 Embeddings Path: embeddings
✅ Successfully connected to S3 bucket: twelvelabs-video-demo-bucket


#### Netflix Open Content

The [Netflix Open Content](https://opencontent.netflix.com/) is an open source content available under the [Creative Commons Attribution 4.0 International Public License](https://www.google.com/url?q=https%3A%2F%2Fcreativecommons.org%2Flicenses%2Fby%2F4.0%2Flegalcode&sa=D&sntz=1&usg=AOvVaw3DDX6ldzWtAO5wOs5KkByf).

The assets are available for download at: http://download.opencontent.netflix.com/

We will be utilizing a subset of the videos for demonstrating how to utilize the TwelveLabs models on Amazon Bedrock.

In [25]:
# Sample video S3 URIs
sample_videos = [
    # 's3://download.opencontent.netflix.com/TechblogAssets/CosmosLaundromat/encodes/CosmosLaundromat_2048x858_24fps_SDR.mp4',
    # 's3://download.opencontent.netflix.com/TechblogAssets/Meridian/encodes/Meridian_3840x2160_5994fps_SDR.mp4',
    's3://download.opencontent.netflix.com/TechblogAssets/Sparks/encodes/Sparks_4096x2160_5994fps_SDR.mp4'
]

In [26]:
# Unsigned S3 client
public_s3_client = boto3.client('s3', config=botocore.client.Config(signature_version=botocore.UNSIGNED))

In [27]:
def parse_s3_uri(s3_uri: str) -> tuple[str, str]:
    """
    Parses an S3 URI like s3://bucket-name/path/to/object and returns (bucket, key)

    Args:
        s3_uri (str): The S3 URI to parse
        
    Returns:
        tuple[str, str]: The bucket and key
    """
    pattern = r'^s3://([^/]+)/(.+)$'
    match = re.match(pattern, s3_uri)
    if not match:
        raise ValueError(f"Invalid S3 URI format: {s3_uri}")
    return match.group(1), match.group(2)


def copy_public_s3_object_to_private_bucket(public_s3_uri: str, dest_bucket: str, dest_key: str, aws_profile: str = 'default') -> None:
    """
    Copies a public S3 object to a private bucket

    Args:
        public_s3_uri (str): The S3 URI of the public object to copy
        dest_bucket (str): The name of the private bucket to copy to
        dest_key (str): The key of the object to copy to
        aws_profile (str): The AWS profile to use for the authenticated client
    """

    # Parse source bucket and key
    source_bucket, source_key = parse_s3_uri(public_s3_uri)

    # Anonymous client to read public object
    anon_s3 = boto3.client('s3', config=botocore.client.Config(signature_version=botocore.UNSIGNED))

    print(f"Downloading from {public_s3_uri}...")
    response = anon_s3.get_object(Bucket=source_bucket, Key=source_key)
    data = response['Body'].read()

    print(f"Uploading to s3://{dest_bucket}/{dest_key} ...")
    s3_client.put_object(Bucket=dest_bucket, Key=dest_key, Body=data)

    print("✅ Copy completed successfully!")

In [28]:
# Copy videos to the S3 bucket
for video_uri in sample_videos:
    # Extract the filename from the S3 key
    _, src_key = parse_s3_uri(video_uri)
    filename = src_key.split("/")[-1]
    dest_key = f"{S3_VIDEOS_PATH}/{filename}"
    copy_public_s3_object_to_private_bucket(
        public_s3_uri=video_uri,
        dest_bucket=S3_BUCKET_NAME,
        dest_key=dest_key
    )

Uploading to s3://twelvelabs-video-demo-bucket/videos/Sparks_4096x2160_5994fps_SDR.mp4 ...
✅ Copy completed successfully!


### Part 2b: Creating vector embeddings with Marengo on Bedrock

#### TwelveLabs Marengo

Marengo is a multimodal embedding model that combines visual, audio, and text analysis for comprehensive video understanding. It supports fine-grained search (brand logos, OCR, small objects), motion detection, and object counting.

#### Marengo Embed 3.0 on Bedrock

- Model ID: `twelvelabs.marengo-embed-3-0-v1:0`
- Input: Video, Text, Audio, Image
- Output: 512-dim embeddings
- Max video: 4 hours (< 6GB)
- Sync API (`InvokeModel`): Text, Image
- Async API (`StartAsyncInvoke`): Video, Audio, Image, Text

**Resources:** [AWS Docs](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-marengo-3.html) | [TwelveLabs Docs](https://docs.twelvelabs.io/v1.3/docs/concepts/models/marengo)

In [29]:
# Marengo model configuration
MARENGO_MODEL_ID = 'twelvelabs.marengo-embed-3-0-v1:0'

In [30]:
MARENGO_INFERENCE_ID_REGIONS = {
    "us-east-1": "us.twelvelabs.marengo-embed-3-0-v1:0",
    "eu-west-1": "eu.twelvelabs.marengo-embed-3-0-v1:0",
    "ap-northeast-2": "apac.twelvelabs.marengo-embed-3-0-v1:0"
}

In [31]:
try:
    MARENGO_INFERENCE_ID = MARENGO_INFERENCE_ID_REGIONS[AWS_REGION]
    print(MARENGO_INFERENCE_ID)
except KeyError:
    raise ValueError(f"Marengo is not supported for {AWS_REGION}")

us.twelvelabs.marengo-embed-3-0-v1:0


##### Creating a text embedding with Marengo InvokeModel API

In [32]:
# Wrapper function to create text embedding
def create_text_embedding(text_query: str) -> list:
    """
    Create embeddings for text using Marengo on Bedrock

    Args:
        text_query (str): The text query to create an embedding for
        
    Returns:
        list: A list of embedding data
    """
    
    model_input = { 
        "inputType": "text",
        "text": {
            "inputText": text_query
        }
    }

    response = bedrock_client.invoke_model(
        modelId=MARENGO_INFERENCE_ID,
        body=json.dumps(model_input)
    )
    
    embedding_data = json.loads(response['body'].read().decode('utf-8'))['data']
    
    return embedding_data

In [33]:
# Example: Create text embedding
text_query = "two people having a conversation in a car"

print(f"Creating text embedding for query")
text_embedding_data = create_text_embedding(text_query)

print(f"✅ Text embedding created successfully with {len(text_embedding_data)} segment and {len(text_embedding_data[0]['embedding'])} dimensions.")

Creating text embedding for query
✅ Text embedding created successfully with 1 segment and 512 dimensions.


##### Creating an image embedding with Marengo InvokeModel API

In [34]:
# Choose image
image_path = "assets/images/image.jpg"

In [35]:
# Wrapper function to create image embedding
def create_image_embedding(image_path: str) -> list:
    """
    Create embeddings for image using Marengo on Bedrock
    
    Args:
        image_path (str): The path to the image to create an embedding for
        
    Returns:
        list: A list of embedding data
    """

    # Convert image to base64
    with open(image_path, "rb") as image_file:
        image_data = image_file.read()
        image_base64 = base64.b64encode(image_data).decode('utf-8')

    model_input = { 
        "inputType": "image",
        "image": {
            "mediaSource": {
                "base64String": image_base64
            }
        }
    }

    response = bedrock_client.invoke_model(
        modelId=MARENGO_INFERENCE_ID,
        body=json.dumps(model_input)
    )
    
    embedding_data = json.loads(response['body'].read().decode('utf-8'))['data']
    
    return embedding_data

In [36]:
# Example: Create image embedding
print(f"Creating embeddings for image at {image_path}")
image_embedding_data = create_image_embedding(image_path)

print(f"✅ Image embedding created successfully with {len(image_embedding_data)} segment(s)")

Creating embeddings for image at assets/images/image.jpg
✅ Image embedding created successfully with 1 segment(s)


##### Creating a text + image embedding with Marengo InvokeModel API

In [37]:
# Wrapper function to create text_image embedding
def create_text_image_embedding(text_query: str, image_path: str) -> list:
    """
    Create embeddings for image using Marengo on Bedrock
    
    Args:
        text_query (str): The text query to create an embedding for
        image_path (str): The path to the image to create an embedding for
        
    Returns:
        list: A list of embedding data
    """

    # Convert image to base64
    with open(image_path, "rb") as image_file:
        image_data = image_file.read()
        image_base64 = base64.b64encode(image_data).decode('utf-8')

    model_input = { 
        "inputType": "text_image",
        "text_image": {
            "inputText": text_query,
            "mediaSource": {
                "base64String": image_base64
            }
        }
    }

    response = bedrock_client.invoke_model(
        modelId=MARENGO_INFERENCE_ID,
        body=json.dumps(model_input)
    )
    
    embedding_data = json.loads(response['body'].read().decode('utf-8'))['data']
    
    return embedding_data

In [38]:
# Example: Create text_image embedding
print(f"Creating embeddings for image at {image_path}")
image_embedding_data = create_text_image_embedding(text_query, image_path)

print(f"✅ Text + image embedding created successfully with {len(image_embedding_data)} segment(s)")

Creating embeddings for image at assets/images/image.jpg
✅ Text + image embedding created successfully with 1 segment(s)


##### Creating a video embedding with Marengo StartAsyncInvoke API

**Video** and **audio** inputs can be processed by Marengo with the [StartAsyncInvoke API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_StartAsyncInvoke.html). The model outputs will land in the S3 location specified by `outputDataConfig`.

In [39]:
# Create an empty dictionary to store video embedding to video file mapping
video_embedding_mapping = {}

Since the StartAsyncInvoke API asynchronously executes the task, the helper function below triggers the task and waits for it to complete. It then retrieves the outputs from the output S3 location.

In [40]:
# Helper function to wait for async embedding results
def wait_for_embedding_output(s3_bucket: str, s3_prefix: str, invocation_arn: str, verbose: bool = False) -> list:
    """
    Wait for Bedrock async embedding task to complete and retrieve results

    Args:
        s3_bucket (str): The S3 bucket name
        s3_prefix (str): The S3 prefix for the embeddings
        invocation_arn (str): The ARN of the Bedrock async embedding task

    Returns:
        list: A list of embedding data
        
    Raises:
        Exception: If the embedding task fails or no output.json is found
    """
    
    # Wait until task completes
    status = None
    while status not in ["Completed", "Failed", "Expired"]:
        response = bedrock_client.get_async_invoke(invocationArn=invocation_arn)
        status = response['status']
        if verbose:
            clear_output(wait=True)
            print(f"Embedding task status: {status}")
        time.sleep(5)
    
    if status != "Completed":
        raise Exception(f"Embedding task failed with status: {status}")
    
    # Retrieve the output from S3
    response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)
    
    for obj in response.get('Contents', []):
        if obj['Key'].endswith('output.json'):
            output_key = obj['Key']
            obj = s3_client.get_object(Bucket=s3_bucket, Key=output_key)
            content = obj['Body'].read().decode('utf-8')
            data = json.loads(content).get("data", [])
            return data
    
    raise Exception("No output.json found in S3 prefix")

In [41]:
# Wrapper function to create and retrieve video embeddings
def create_video_embedding(video_s3_uri: str) -> list:
    """
    Create embeddings for video using Marengo on Bedrock
    
    Args:
        video_s3_uri (str): The S3 URI of the video to create an embedding for
        
    Returns:
        list: A list of embedding data
    """
    
    unique_id = uuid.uuid4()
    s3_output_prefix = f'{S3_EMBEDDINGS_PATH}/{S3_VIDEOS_PATH}/{unique_id}'
    
    response = bedrock_client.start_async_invoke(
        modelId=MARENGO_MODEL_ID,
        modelInput={
            "inputType": "video",
            "video": {
                "mediaSource": {
                    "s3Location": {
                        "uri": video_s3_uri,
                        "bucketOwner": aws_account_id
                    }
                },
                "embeddingOption": ["visual"],
                "embeddingScope": ["clip"]
            }
        },
        outputDataConfig={
            "s3OutputDataConfig": {
                "s3Uri": f's3://{S3_BUCKET_NAME}/{s3_output_prefix}'
            }
        }
    )
    
    invocation_arn = response["invocationArn"]
    print(f"Video embedding task started: {invocation_arn}")
    
    # Wait for completion and get results
    try:
        embedding_data = wait_for_embedding_output(S3_BUCKET_NAME, s3_output_prefix, invocation_arn)
        video_embedding_mapping[str(unique_id)] = video_s3_uri
    except Exception as e:
        print(f"Error waiting for embedding output: {e}")
        return None
    
    return embedding_data


In [44]:
# Example: Create video embeddings
videos = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=S3_VIDEOS_PATH)["Contents"]
video_uri = f"s3://{S3_BUCKET_NAME}/{videos[0]['Key']}"

print(f"Creating embeddings for video: {video_uri}")
video_embedding_data = create_video_embedding(video_uri)

print(f"✅ Video embedding created successfully with {len(video_embedding_data)} segment(s)")

Creating embeddings for video: s3://twelvelabs-video-demo-bucket/videos/Sparks_4096x2160_5994fps_SDR.mp4


ValidationException: An error occurred (ValidationException) when calling the StartAsyncInvoke operation: Invalid S3 credentials

## Part 3: Video Search with Marengo and Vector Database

The Marengo embeddings can be stored in a vector database to power vector retrieval architecture for images, audios, and videos.

In this section, we will deploy a vector database and create an index to store the Marengo embeddings. There are 2 options for the vector database in this workshop:
- (Option 1) Part 3a: [Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/)
- (Option 2) Part 3b: [Amazon S3 Vectors](https://aws.amazon.com/s3/features/vectors/)

If you are attending an instructor-led workshop, please follow the instructor's instructions on which option to use.

#### Bulk process videos in S3 with Marengo

Using the `StartAsyncInvoke` API, let's bulk process the videos to index into a vector database for search. We will be processing all video assets in the S3 bucket with the `/videos` prefix.

In a production architecture, a separate database to store the video mappings is recommended. For simplicity, in this workshop, the video mappings will be stored in the `video_embedding_mapping` dictionary.

In [ ]:
# Clear existing video embeddings in S3 bucket
response = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=f"{S3_EMBEDDINGS_PATH}/{S3_VIDEOS_PATH}")

# Empty video embeddings in S3 bucket
try:
    if 'Contents' in response:
        objects_to_delete = [{'Key': obj['Key']} for obj in response['Contents']]
        s3_client.delete_objects(
            Bucket=S3_BUCKET_NAME,
            Delete={'Objects': objects_to_delete}
        )
        print(f"✅ Removed existing video embeddings successfully.")
    else:
        print(f"✅ No existing video embeddings found.")
except Exception as e:
    print(f"❌ Error emptying video embeddings: {e}")

# Create an empty dictionary to store video embedding to video file mapping
video_embedding_mapping = {}

In [ ]:
# Retrieve the list of videos in the s3 bucket and loop through them to create embeddings
videos = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=S3_VIDEOS_PATH)["Contents"]

for video in videos:
    video_uri = f"s3://{S3_BUCKET_NAME}/{video['Key']}"
    print(f"Creating embeddings for video: {video_uri}")
    video_embedding_data = create_video_embedding(video_uri)

    print(f"✅ Video embedding created successfully with {len(video_embedding_data)} segment(s) from {video['Key']}")

Being able to play a video will allow us to visualize the search results. Let's create a helper function that plays a given video at a specific start time.

In [ ]:
# Helper function to play a video at a specific start time
def play_video(video_url: str, start_time: float) -> None:
    """
    Play a video at a specific start time.

    Args:
        video_url (str): The URL of the video to play.
        start_time (float): The start time of the video in seconds.
    """

    # HTML code for the video player
    html_code = f"""
    <video width="640" controls>
        <source src="{video_url}#t={start_time}" type="video/mp4">
    </video>
    """
    display(HTML(html_code))

### (Option 1) Part 3a: Video Search with Amazon OpenSearch

[Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/) — serverless vector search with millisecond response times.

#### Configure Amazon Opensearch Serverless Client

In [ ]:
# OpenSearch Serverless configuration
OPENSEARCH_ENDPOINT = "<YOUR_OPENSEARCH_ENDPOINT>"  # TODO: Replace with your OpenSearch endpoint
INDEX_NAME = "video-embeddings-index"

# Validate OpenSearch endpoint
if OPENSEARCH_ENDPOINT == "<YOUR_OPENSEARCH_ENDPOINT>" or OPENSEARCH_ENDPOINT == "":
    raise ValueError("Please replace <YOUR_OPENSEARCH_ENDPOINT> with your OpenSearch endpoint")
elif OPENSEARCH_ENDPOINT.startswith("https://"):
    OPENSEARCH_ENDPOINT = OPENSEARCH_ENDPOINT.replace("https://", "")

# Create OpenSearch client for Amazon OpenSearch Serverless
service = "aoss"
credentials = session.get_credentials()
auth = AWSV4SignerAuth(credentials, AWS_REGION, service)

os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_ENDPOINT, "port": 443}],
    http_auth=auth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    pool_maxsize=20,
)

#### Create a new OpenSearch index

In [ ]:
# Create OpenSearch vector index
def create_opensearch_index(os_client: OpenSearch, index_name: str):
    """
    Create a vector index in OpenSearch for storing video embeddings

    Args:
        os_client (OpenSearch): The OpenSearch client
        index_name (str): The name of the index to create

    Returns:
        None
    """
    
    if os_client.indices.exists(index=index_name):
        print(f"Index '{index_name}' already exists.")
        print(f"❌ Set a new index name or delete the existing index.")
        return
    
    index_body = {
        "settings": {
            "index": {
                "knn": True,
                "number_of_shards": 1,
            }
        },
        "mappings": {
            "properties": {
                "embedding": {
                    "type": "knn_vector",
                    "dimension": 512,
                    "method": {
                        "engine": "faiss",
                        "name": "hnsw",
                        "space_type": "cosinesimil",
                    },
                },
                "start_time": {"type": "float"},
                "end_time": {"type": "float"},
                "video_id": {"type": "keyword"},
                "segment_text": {"type": "text"},
                "embedding_option": {"type": "keyword"}
            }
        },
    }
    
    os_client.indices.create(index=index_name, body=index_body)
    print(f"✅ Index '{index_name}' created successfully.")

# Create the index
create_opensearch_index(os_client, INDEX_NAME)


#### Insert embeddings into OpenSearch index

In [ ]:
# Index video embeddings in OpenSearch
def index_video_embeddings(os_client: OpenSearch, index_name: str, video_embeddings: list, video_id: str = "sample_video") -> int:
    """
    Index video embeddings into OpenSearch
    
    Args:
        os_client (OpenSearch): The OpenSearch client
        index_name (str): The name of the index to use
        video_embeddings (list): The list of video embeddings
        video_id (str): The id of the video

    Returns:
        int: The number of documents indexed
    """
    
    documents = []
    
    for i, segment in enumerate(video_embeddings):
        document = {
            "embedding": segment["embedding"],
            "start_time": segment["startSec"],
            "end_time": segment["endSec"],
            "video_id": video_id,
            "segment_id": i,
            "embedding_option": segment.get("embeddingOption", "visual"),
            "embedding_scope": segment.get("embeddingScope", "clip")
        }
        documents.append(document)
    
    # Bulk index documents
    bulk_data = []
    for doc in documents:
        bulk_data.append({"index": {"_index": index_name}})
        bulk_data.append(doc)
    
    # Convert to bulk format
    bulk_body = "\n".join(json.dumps(item) for item in bulk_data) + "\n"
    
    response = os_client.bulk(body=bulk_body, index=index_name)
    
    if response["errors"]:
        print("Some documents failed to index:")
        for item in response["items"]:
            if "index" in item and "error" in item["index"]:
                print(f"Error: {item['index']['error']}")
    
    return len(documents)

In [ ]:
# Retrieve the list of embedding files in the S3 bucket
embedding_files = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=f"{S3_EMBEDDINGS_PATH}/{S3_VIDEOS_PATH}").get("Contents", [])

for embedding_file in embedding_files:
    embedding_key = embedding_file["Key"]
    if not embedding_key.endswith("output.json"):
        continue  # Skip non-JSON files

    embedding_obj = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=embedding_key)
    content = embedding_obj['Body'].read().decode('utf-8')
    embedding_data = json.loads(content).get("data", [])

    # Use the index_video_embeddings function to index the embedding data into OpenSearch
    num_indexed = index_video_embeddings(os_client, INDEX_NAME, embedding_data, video_id=embedding_key.split("/")[2])

    print(f"✅ Indexed {num_indexed} segments from {embedding_key}")

#### Query OpenSearch index with text

In [ ]:
# Text Query Search Function
def search_opensearch_by_text(query_text: str, top_k: int=5) -> list:
    """
    Search for video segments using text queries

    Args:
        query_text (str): The text query to search for.
        top_k (int): The number of videos to return.

    Returns:
        list: A list of video segments that match the query.
    """
    
    # Generate embedding for the text query
    print(f"Generating embedding for query: '{query_text}'")
    query_embedding_data = create_text_embedding(query_text)
    query_embedding = query_embedding_data[0]["embedding"]
    
    # Search OpenSearch index
    search_body = {
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": top_k
                }
            }
        },
        "size": top_k,
        "_source": ["start_time", "end_time", "video_id", "segment_id"]
    }
    
    response = os_client.search(index=INDEX_NAME, body=search_body)
    
    print(f"\n✅ Found {len(response['hits']['hits'])} matching segments:")
    results = []
    
    for hit in response['hits']['hits']:
        result = {
            "score": hit["_score"],
            "video_id": hit["_source"]["video_id"],
            "segment_id": hit["_source"]["segment_id"],
            "start_time": hit["_source"]["start_time"],
            "end_time": hit["_source"]["end_time"]
        }
        results.append(result)
        
        print(f"  Score: {result['score']:.4f} | Video: {video_embedding_mapping[result['video_id']]} | "
              f"Segment: {result['segment_id']} | Time: {result['start_time']:.1f}s - {result['end_time']:.1f}s")
    
    return results


In [ ]:
text_query = "a person wearing safety gear and welding with a forest in the background"

In [ ]:
# Example text search
text_search_results = search_opensearch_by_text(text_query, top_k=3)

In [ ]:
# View top result
top_text_result = text_search_results[0]
video_bucket, video_key = parse_s3_uri(video_embedding_mapping[top_text_result["video_id"]])

# Generate presigned URL for the video
presigned_url = s3_client.generate_presigned_url(
    "get_object",
    Params={"Bucket": video_bucket, "Key": video_key},
    ExpiresIn=3600
)

In [ ]:
# Set the video stream URL and the start time
video_url = presigned_url
start_time = top_text_result["start_time"]
print(f"\nVideo URL: {video_url}")
print(f"Start time: {start_time}")

# Play the video
play_video(video_url, start_time)

#### Query OpenSearch index with image

In [ ]:
# Image Query Search Function
def search_opensearch_by_image(image_path: str, top_k: int=5) -> list:
    """
    Search for videos that contain the given image.

    Args:
        image_path (str): The path to the image to search for.
        top_k (int): The number of videos to return.

    Returns:
        list: A list of video segments that match the query.
    """

    # Create image embedding
    print(f"Creating embeddings for image: {image_path}")
    embedding_data = create_image_embedding(image_path)
    query_embedding = embedding_data[0]["embedding"]

    # Search OpenSearch index
    search_body = {
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": top_k
                }
            }
        },
        "size": top_k,
        "_source": ["start_time", "end_time", "video_id", "segment_id"]
    }
    
    response = os_client.search(index=INDEX_NAME, body=search_body)
    
    print(f"\n✅ Found {len(response['hits']['hits'])} matching segments:")
    results = []
    
    for hit in response['hits']['hits']:
        result = {
            "score": hit["_score"],
            "video_id": hit["_source"]["video_id"],
            "segment_id": hit["_source"]["segment_id"],
            "start_time": hit["_source"]["start_time"],
            "end_time": hit["_source"]["end_time"]
        }
        results.append(result)
        
        print(f"  Score: {result['score']:.4f} | Video: {video_embedding_mapping[result['video_id']]} | "
              f"Segment: {result['segment_id']} | Time: {result['start_time']:.1f}s - {result['end_time']:.1f}s")
    
    return results

In [ ]:
image_query = "assets/images/image.jpg"

display(Image(filename=image_query, width=200))

In [ ]:
# Example image search
image_search_results = search_opensearch_by_image(image_path=image_query, top_k=3)

In [ ]:
# View top result
top_image_result = image_search_results[0]
video_bucket, video_key = parse_s3_uri(video_embedding_mapping[top_image_result["video_id"]])

# Generate presigned URL for the video
presigned_url = s3_client.generate_presigned_url(
    "get_object",
    Params={"Bucket": video_bucket, "Key": video_key},
    ExpiresIn=3600
)

In [ ]:
# Set the video stream URL and the start time
video_url = presigned_url
start_time = top_image_result["start_time"]
print(f"\nVideo URL: {video_url}")
print(f"Start time: {start_time}")

# Play the video
play_video(video_url, start_time)

#### Query OpenSearch index with Text + Image (Composed Image Query)

In [ ]:
# Composed Image Query Search Function
def search_opensearch_by_text_image(text_query: str, image_path: str, top_k: int=5) -> list:
    """
    Search for videos that contain the given image.

    Args:
        text_query (str): The text query to search for.
        image_path (str): The path to the image to search for.
        top_k (int): The number of videos to return.

    Returns:
        list: A list of video segments that match the query.
    """

    # Create text_image embedding
    print(f"Creating embeddings for text + image: {text_query} and {image_path}")
    embedding_data = create_text_image_embedding(text_query, image_path)
    query_embedding = embedding_data[0]["embedding"]

    # Search OpenSearch index
    search_body = {
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": top_k
                }
            }
        },
        "size": top_k,
        "_source": ["start_time", "end_time", "video_id", "segment_id"]
    }
    
    response = os_client.search(index=INDEX_NAME, body=search_body)
    
    print(f"\n✅ Found {len(response['hits']['hits'])} matching segments:")
    results = []
    
    for hit in response['hits']['hits']:
        result = {
            "score": hit["_score"],
            "video_id": hit["_source"]["video_id"],
            "segment_id": hit["_source"]["segment_id"],
            "start_time": hit["_source"]["start_time"],
            "end_time": hit["_source"]["end_time"]
        }
        results.append(result)
        
        print(f"  Score: {result['score']:.4f} | Video: {video_embedding_mapping[result['video_id']]} | "
              f"Segment: {result['segment_id']} | Time: {result['start_time']:.1f}s - {result['end_time']:.1f}s")
    
    return results

In [ ]:
text_query = "a person in an elevator"

In [ ]:
image_query = "assets/images/image.jpg"

display(Image(filename=image_query, width=200))

In [ ]:
# Example text + image search
text_image_search_results = search_opensearch_by_text_image(text_query=text_query, image_path=image_query, top_k=3)

In [ ]:
# View top result
top_text_image_result = text_image_search_results[0]
video_bucket, video_key = parse_s3_uri(video_embedding_mapping[top_text_image_result["video_id"]])

# Generate presigned URL for the video
presigned_url = s3_client.generate_presigned_url(
    "get_object",
    Params={"Bucket": video_bucket, "Key": video_key},
    ExpiresIn=3600
)

In [ ]:
# Set the video stream URL and the start time
video_url = presigned_url
start_time = top_text_image_result["start_time"]
print(f"\nVideo URL: {video_url}")
print(f"Start time: {start_time}")

# Play the video
play_video(video_url, start_time)

### (Option 2) Part 3b: Video Search with S3 Vectors

[Amazon S3 Vectors](https://aws.amazon.com/s3/features/vectors/) — cost-optimized vector storage with native S3 integration and sub-second query performance.

#### Configure S3 Vectors Client

In [ ]:
# S3 Vector Bucket Configuration
S3_VECTOR_BUCKET_NAME = "<YOUR_S3_VECTOR_BUCKET_NAME>" # TODO: Replace with your S3 vector bucket name
S3_VECTOR_INDEX_NAME = "my-vector-index"

# Validate S3 Vector bucket and index names
if S3_VECTOR_BUCKET_NAME == "<YOUR_S3_VECTOR_BUCKET_NAME>" or S3_VECTOR_BUCKET_NAME == "":
    raise ValueError("Please replace <YOUR_S3_VECTOR_BUCKET_NAME> with your S3 vector bucket name")

# S3 Vectors Client
s3_vectors_client = session.client("s3vectors")

In [ ]:
# Verify S3 Vector bucket access
try:
    s3_vectors_client.get_vector_bucket(vectorBucketName=S3_VECTOR_BUCKET_NAME)
    print(f"✅ Successfully connected to S3 Vector bucket: {S3_VECTOR_BUCKET_NAME}")
except Exception as e:
    print(f"❌ Error connecting to S3 Vector bucket: {e}")
    print(f"Please ensure the vector bucket exists and you have proper permissions.")

#### Create a new S3 Vectors index

In [ ]:
# Configure S3 Vector index
try:
    s3_vectors_client.create_index(
        vectorBucketName=S3_VECTOR_BUCKET_NAME, 
        indexName=S3_VECTOR_INDEX_NAME,
        dataType='float32',
        dimension=512,
        distanceMetric='cosine'
    )
    print(f"✅ Index '{S3_VECTOR_INDEX_NAME}' created successfully.")
except s3_vectors_client.exceptions.ConflictException as e:
    print(f'Index {S3_VECTOR_INDEX_NAME} already exists')
    print(f"❌ Set a new index name or delete the existing index.")
except Exception as e:
    print(f"❌ Error creating index: {e}")

#### Insert embeddings into S3 Vectors index

In [ ]:
def index_video_embeddings(s3_vector_bucket: str, s3_vector_index: str, video_embeddings: list, video_id: str = "sample_video") -> int:
    """
    Index video embeddings into S3 Vectors
    
    Args:
        s3_vector_bucket (str): The name of the bucket to use
        s3_vector_index (str): The name of the index to use
        video_embeddings (list): The list of video embeddings
        video_id (str): The id of the video

    Returns:
        int: The number of documents indexed
    """

    embeddings = []
    for i, segment in enumerate(video_embeddings):
        embeddings.append({
            "key": f'{segment["embeddingOption"]} {segment["startSec"]} {segment["endSec"]}',
            "data": {"float32": segment["embedding"]},
            "metadata": {
                "start_time": segment["startSec"], 
                "end_time": segment["endSec"],
                "video_id": video_id,
                "segment_id": i,
                "embedding_option": segment.get("embeddingOption", "visual"),
                "embedding_scope": segment.get("embeddingScope", "clip")
            }
        })

    # Write embeddings into vector index with metadata.
    s3_vectors_client.put_vectors(
        vectorBucketName=s3_vector_bucket,   
        indexName=s3_vector_index,   
        vectors=embeddings
    )

    return len(embeddings)

In [ ]:
# Retrieve the list of embedding files in the S3 bucket
embedding_files = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=f"{S3_EMBEDDINGS_PATH}/{S3_VIDEOS_PATH}").get("Contents", [])

for embedding_file in embedding_files:
    embedding_key = embedding_file["Key"]
    if not embedding_key.endswith("output.json"):
        continue  # Skip non-JSON files

    embedding_obj = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=embedding_key)
    content = embedding_obj['Body'].read().decode('utf-8')
    embedding_data = json.loads(content).get("data", [])

    # Use the index_video_embeddings function to index the embedding data into S3 Vectors
    num_indexed = index_video_embeddings(S3_VECTOR_BUCKET_NAME, S3_VECTOR_INDEX_NAME, embedding_data, video_id=embedding_key.split("/")[2])

    print(f"✅ Indexed {num_indexed} segments from {embedding_key}")

#### Query S3 Vectors index with text

In [ ]:
# Text Query Search Function
def search_s3_vectors_by_text(query_text: str, top_k: int=5) -> list:
    """
    Search for video segments using text queries

    Args:
        query_text (str): The text query to search for.
        top_k (int): The number of videos to return.

    Returns:
        list: A list of video segments that match the query.
    """
    # Generate embedding for the text query
    print(f"Generating embedding for query: '{query_text}'")
    query_embedding_data = create_text_embedding(query_text)
    query_embedding = query_embedding_data[0]["embedding"]

    # Search S3 Vector Index
    response = s3_vectors_client.query_vectors(
        vectorBucketName=S3_VECTOR_BUCKET_NAME,
        indexName=S3_VECTOR_INDEX_NAME,
        queryVector={"float32": query_embedding}, 
        topK=top_k, 
        returnDistance=True,
        returnMetadata=True
    )

    print(f"\n✅ Found {len(response['vectors'])} matching segments:")
    results = []
    
    for hit in response['vectors']:
        result = {
            "score": hit["distance"],
            "video_id": hit["metadata"]['video_id'],
            "segment_id": hit["metadata"]["segment_id"],
            "start_time": hit["metadata"]["start_time"],
            "end_time": hit["metadata"]["end_time"]
        }
        results.append(result)
        
        print(f"  Score: {result['score']:.4f} | Video: {video_embedding_mapping[result['video_id']]} | "
              f"Segment: {result['segment_id']} | Time: {result['start_time']:.1f}s - {result['end_time']:.1f}s")
    
    return results

In [ ]:
text_query = "a person wearing safety gear and welding with a forest in the background"

In [ ]:
# Example text search
text_search_results = search_s3_vectors_by_text(text_query, top_k=3)

In [ ]:
# View top result
top_text_result = text_search_results[0]
video_bucket, video_key = parse_s3_uri(video_embedding_mapping[top_text_result["video_id"]])

# Generate presigned URL for the video
presigned_url = s3_client.generate_presigned_url(
    "get_object",
    Params={"Bucket": video_bucket, "Key": video_key},
    ExpiresIn=3600
)

In [ ]:
# Set the video stream URL and the start time
video_url = presigned_url
start_time = top_text_result["start_time"]
print(f"\nVideo URL: {video_url}")
print(f"Start time: {start_time}")

# Play the video
play_video(video_url, start_time)

#### Query S3 Vectors index with image

In [ ]:
# Image Query Search Function
def search_s3_vectors_by_image(image_path: str, top_k: int=5) -> list:
    """
    Search for videos that contain the given image.

    Args:
        image_path (str): The path to the image to search for.
        top_k (int): The number of videos to return.

    Returns:
        list: A list of video segments that match the query.
    """
    # Generate embedding for the image
    print(f"Generating embedding for query: '{image_path}'")
    query_embedding_data = create_image_embedding(image_path)
    query_embedding = query_embedding_data[0]["embedding"]

    # Search S3 Vector Index
    response = s3_vectors_client.query_vectors(
        vectorBucketName=S3_VECTOR_BUCKET_NAME,
        indexName=S3_VECTOR_INDEX_NAME,
        queryVector={"float32": query_embedding}, 
        topK=top_k, 
        returnDistance=True,
        returnMetadata=True
    )

    print(f"\n✅ Found {len(response['vectors'])} matching segments:")
    results = []
    
    for hit in response['vectors']:
        result = {
            "score": hit["distance"],
            "video_id": hit["metadata"]['video_id'],
            "segment_id": hit["metadata"]["segment_id"],
            "start_time": hit["metadata"]["start_time"],
            "end_time": hit["metadata"]["end_time"]
        }
        results.append(result)
        
        print(f"  Score: {result['score']:.4f} | Video : {result['video_id']}"
              f"Segment: {result['segment_id']} | Time: {result['start_time']:.1f}s - {result['end_time']:.1f}s")

    return results

In [ ]:
image_query = "assets/images/image.jpg"

display(Image(filename=image_query, width=200))

In [ ]:
# Example image search
image_search_results = search_s3_vectors_by_image(image_path=image_query, top_k=3)

In [ ]:
# View top result
top_image_result = image_search_results[0]
video_bucket, video_key = parse_s3_uri(video_embedding_mapping[top_text_result["video_id"]])

# Generate presigned URL for the video
presigned_url = s3_client.generate_presigned_url(
    "get_object",
    Params={"Bucket": video_bucket, "Key": video_key},
    ExpiresIn=3600
)

In [ ]:
# Set the video stream URL and the start time
video_url = presigned_url
start_time = top_image_result["start_time"]
print(f"\nVideo URL: {video_url}")
print(f"Start time: {start_time}")

# Play the video
play_video(video_url, start_time)

#### Query S3 Vectors index with Text + Image (Composed Image Query)

In [ ]:
# Composed Image Query Search Function
def search_s3_vectors_by_text_image(text_query: str, image_path: str, top_k: int=5) -> list:
    """
    Search for videos that contain the given image.

    Args:
        text_query (str): The text query to search for.
        image_path (str): The path to the image to search for.
        top_k (int): The number of videos to return.

    Returns:
        list: A list of video segments that match the query.
    """

    # Create text_image embedding
    print(f"Creating embeddings for text + image: {text_query} and {image_path}")
    embedding_data = create_text_image_embedding(text_query, image_path)
    query_embedding = embedding_data[0]["embedding"]

    # Search S3 Vector Index
    response = s3_vectors_client.query_vectors(
        vectorBucketName=S3_VECTOR_BUCKET_NAME,
        indexName=S3_VECTOR_INDEX_NAME,
        queryVector={"float32": query_embedding}, 
        topK=top_k, 
        returnDistance=True,
        returnMetadata=True
    )
    
    print(f"\n✅ Found {len(response['vectors'])} matching segments:")
    results = []
    
    for hit in response['vectors']:
        result = {
            "score": hit["distance"],
            "video_id": hit["metadata"]['video_id'],
            "segment_id": hit["metadata"]["segment_id"],
            "start_time": hit["metadata"]["start_time"],
            "end_time": hit["metadata"]["end_time"]
        }
        results.append(result)
        
        print(f"  Score: {result['score']:.4f} | Video : {result['video_id']}"
              f"Segment: {result['segment_id']} | Time: {result['start_time']:.1f}s - {result['end_time']:.1f}s")

    return results

In [ ]:
text_query = "a person in an elevator"

In [ ]:
image_query = "assets/images/image.jpg"

display(Image(filename=image_query, width=200))

In [ ]:
# Example text + image search
text_image_search_results = search_s3_vectors_by_text_image(text_query=text_query, image_path=image_query, top_k=3)

In [ ]:
# View top result
top_text_image_result = text_image_search_results[0]
video_bucket, video_key = parse_s3_uri(video_embedding_mapping[top_text_image_result["video_id"]])

# Generate presigned URL for the video
presigned_url = s3_client.generate_presigned_url(
    "get_object",
    Params={"Bucket": video_bucket, "Key": video_key},
    ExpiresIn=3600
)

In [ ]:
# Set the video stream URL and the start time
video_url = presigned_url
start_time = top_text_image_result["start_time"]
print(f"\nVideo URL: {video_url}")
print(f"Start time: {start_time}")

# Play the video
play_video(video_url, start_time)

---

---
## Cleanup


#### Delete OpenSearch Index

In [ ]:
# Delete OpenSearch index
try:
    response = os_client.indices.delete(
        index=INDEX_NAME
    )
    print(f"✅ Index '{INDEX_NAME}' successfully deleted.")
except Exception as e:
    print(f"❌ Error deleting index '{INDEX_NAME}': {e}")

#### Delete S3 Vectors Index

In [ ]:
# Delete S3 Vectors index
try:
    response = s3_vectors_client.delete_index(
        vectorBucketName=S3_VECTOR_BUCKET_NAME,
        indexName=S3_VECTOR_INDEX_NAME
    )
    print(f"✅ Index '{S3_VECTOR_INDEX_NAME}' successfully deleted.")
except Exception as e:
    print(f"❌ Error deleting index '{S3_VECTOR_INDEX_NAME}': {e}")

#### Empty S3 bucket

In [ ]:
# List objects and prepare for deletion
response = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME)

# Empty S3 bucket
try:
    if 'Contents' in response:
        objects_to_delete = [{'Key': obj['Key']} for obj in response['Contents']]
        s3_client.delete_objects(
            Bucket=S3_BUCKET_NAME,
            Delete={'Objects': objects_to_delete}
        )
        print(f"✅ Bucket '{S3_BUCKET_NAME}' emptied successfully.")
    else:
        print(f"✅ Bucket '{S3_BUCKET_NAME}' is already empty.")
except Exception as e:
    print(f"❌ Error emptying bucket '{S3_BUCKET_NAME}': {e}")